# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
price_before = df['price'].str.replace('$', '')
price_before = price_before.astype(float)
revenue_start = (df['qty'] * price_before).sum()
print('revenue before cleaning:', round(revenue_start, 2))

removed = df.duplicated().sum()
df = df.drop_duplicates().copy()
log('duplicates', 'dropped duplicates', removed)
print('rows after:', len(df))

revenue before cleaning: 4852.5
[duplicates] dropped duplicates (15 row(s))
rows after: 300


Explanation: I saved the starting revenue of 4852.50 first so I could compare later, then dropped 15 duplicate rows and was left with 300 rows.

### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].str.replace('$', '')
df['price'] = df['price'].astype(float)
log('price', 'turned price from text into a number', len(df))

revenue_after_dupes = (df['qty'] * df['price']).sum()
print('price type:', df['price'].dtype)
print('revenue now:', round(revenue_after_dupes, 2))

[price] turned price from text into a number (300 row(s))
price type: float64
revenue now: 4594.5


Explanation: Price was text because some values had dollar signs, so I removed them and turned the column into numbers.

### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

missing = df['qty'].isna().sum()
negative = (df['qty'] < 0).sum()
print('missing qty:', missing)
print('negative qty:', negative)

df = df[df['qty'] >= 1].copy()
log('qty', 'dropped rows with missing and rows with negative quantity', missing + negative)
print('rows after dropping missing and negative quantity:', len(df))

revenue_after_qty = (df['qty'] * df['price']).sum()
print('revenue now:', round(revenue_after_qty, 2))

missing qty: 12
negative qty: 13
[qty] dropped rows with missing and rows with negative quantity (25 row(s))
rows after dropping missing and negative quantity: 275
revenue now: 4740.0


Explanation: I found 12 rows with no quantity and 13 with a negative quantity and dropped both, which left 275 rows and changed revenue to 4740.00.

### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'cheeseburger',
    'cheese burger': 'cheeseburger',
    'Foam Finger': 'foam finger',
    'foam finger': 'foam finger',
    'Rain Poncho': 'rain poncho',
    'rain poncho': 'rain poncho',
}

df['item'] = df['item'].replace(ITEM_MAP)
log('item', 'combined 6 different item spellings into 3 uniform spellings', len(df))
print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] combined 6 different item spellings into 3 uniform spellings (275 row(s))
item
foam finger     97
rain poncho     91
cheeseburger    87
Name: count, dtype: int64


Explanation: I made a dictionary that turns all 6 item spellings into 3 product names so the same product is not counted as three different ones.

### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food': 'food',
    'food': 'food',
    'Merch': 'merch',
    'Apparel': 'apparel',
    'RainGear': 'raingear',
    'rain-gear': 'raingear',
}

df['category'] = df['category'].replace(CATEGORY_MAP)
log('category', 'combined 6 different label spellings into 4 uniform labels, kept Apparel separate from Merch', len(df))
print(df['category'].value_counts())

category
Merch        51
Food         51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] combined 6 different label spellings into 4 uniform labels, kept Apparel separate from Merch (275 row(s))
category
food        95
raingear    86
merch       51
apparel     43
Name: count, dtype: int64


Explanation: I did the same for category but kept Apparel separate from Merch, since items that count as Merch are different to items that count as Apparel. For example, a hoodie would belong in a different category and would be stocked differently from a team banner. 

### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].nunique() == 3
assert df['category'].nunique() == 4
print('clean:', df.shape)

clean: (275, 5)


Explanation: I added 2 assertions that check for number of unique items in item and category. All the checks passed. 

### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

by_category = df.groupby('category')['revenue'].sum()
by_category = round(by_category.sort_values(ascending=False), 2)

print(by_category)
print('total revenue:', round(df['revenue'].sum(), 2))

category
food        1656.0
raingear    1512.0
merch        856.5
apparel      715.5
Name: revenue, dtype: float64
total revenue: 4740.0


Explanation: I multiplied quantity by price and grouped by category, which shows food as the top category at 1656.00 dollars out of a 4740.00 total.

**What I would tell the vendor:** _I would tell the vendor to stock more food since it brought in a lot more revenue than the other categories._

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped duplicates,15
1,price,turned price from text into a number,300
2,qty,dropped rows with missing and rows with negati...,25
3,item,combined 6 different item spellings into 3 uni...,275
4,category,combined 6 different label spellings into 4 un...,275


Explanation: The log shows all five cleaning steps in order with how many rows each one changed.

### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Dropping the duplicate rows changed the total the most. Revenue was 4852.50 before that step and 4594.50 after, so the step resulted in a drop of 258.00.

b) I kept Apparel separate from Merch, but I could have merged them since apparel items could technically fall under the merchandise umbrella. Merging would not change my total of 4740.00, but it would change what I report per category, since merch would jump to 1572.00 and pass rain gear at 1512.00 for second place. I kept them separate because the two are stocked differently.